## AGN Project: *Characterization of Galaxies via Optical and Infrared Surveys*

In [1]:
%%capture

%matplotlib inline
%config InlineBackend.figure_formats = ['svg']

import pandas as pd
import numpy as np
import matplotlib as mpl

import matplotlib.pyplot as plt

plt.rcParams['figure.constrained_layout.use'] = True
plt.rcParams['legend.frameon'] = False
plt.rcParams['xtick.minor.visible'] = True
plt.rcParams['ytick.minor.visible'] = True
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['xtick.top'] = True
plt.rcParams['ytick.right'] = True
plt.rcParams['axes.formatter.use_mathtext'] = True

```sql
SELECT TOP 10000
    g.sii_6717_flux, g.sii_6731_flux, g.nii_6584_flux, g.oi_6300_flux, g.oiii_5007_flux,
    g.h_alpha_flux, g.h_beta_flux,
    g.h_alpha_eqw, g.nii_6584_eqw, g.oiii_sigma,
    w.w1mpro, w.w2mpro, w.w3mpro,
    s.z, s.class, s.subclass, s.bestobjid
FROM SpecObj AS s
    JOIN galSpecLine AS g ON s.specobjid = g.specobjid
    JOIN PhotoTag AS p ON s.bestobjid = p.objid
    OUTER APPLY (   SELECT TOP 1 x.wise_cntr
                    FROM wise_xmatch AS x
                    WHERE p.objid = x.sdss_objid AND x.match_dist < 3
                    ORDER BY x.match_dist ASC
                ) AS y
    LEFT JOIN wise_allsky AS w ON y.wise_cntr = w.cntr
WHERE s.class = 'GALAXY' AND s.subclass != 'BROADLINE'
    AND s.snmedian_r > 5.0
    AND s.z < 0.35
    AND g.sii_6717_eqw < 0
    AND g.sii_6731_eqw < 0
    AND g.nii_6584_eqw < 0
    AND g.oi_6300_eqw < 0
    AND g.oiii_5007_eqw < 0
    AND g.h_alpha_eqw < 0
    AND g.h_beta_eqw < 0
    AND (2.355 * g.sigma_forbidden) < 500
    AND (2.355 * g.sigma_balmer) < 500
    AND g.sii_6717_flux > 3 * g.sii_6717_flux_err
    AND g.sii_6731_flux > 3 * g.sii_6731_flux_err
    AND g.nii_6584_flux > 3 * g.nii_6584_flux_err
    AND g.oi_6300_flux > 3 * g.oi_6300_flux_err
    AND g.oiii_5007_flux > 3 * g.oiii_5007_flux_err
    AND g.h_alpha_flux > 3 * g.h_alpha_flux_err
    AND g.h_beta_flux > 3 * g.h_beta_flux_err
ORDER BY s.specobjid ASC
```

In [2]:
df = pd.read_csv('data.csv', skiprows=1)

counts = df['bestobjid'].value_counts()

print(f'\nFound a Total of {len(df)} SDSS Objects, with {len(df.dropna(subset=['w1mpro', 'w2mpro', 'w3mpro']))} Matches to the WISE Database.',
      f'There are {len(counts[counts > 1])} Duplicates in this Table.\n')

df.head(15).style.hide()


Found a Total of 10000 SDSS Objects, with 9800 Matches to the WISE Database. There are 0 Duplicates in this Table.



sii_6717_flux,sii_6731_flux,nii_6584_flux,oi_6300_flux,oiii_5007_flux,h_alpha_flux,h_beta_flux,h_alpha_eqw,nii_6584_eqw,oiii_sigma,w1mpro,w2mpro,w3mpro,z,class,subclass,bestobjid
35.334970,21.657780,51.591550,8.490424,21.444370,99.400050,25.640470,-5.130115,-2.528958,182.796000,13.538000,13.469000,11.074000,0.064656,GALAXY,STARFORMING,1237650795146445031
30.269960,25.722970,25.789630,6.269853,35.466710,127.305800,38.332260,-21.187690,-3.849116,5.724757,15.291000,15.156000,12.344000,0.052654,GALAXY,STARFORMING,1237648720142401670
36.779390,37.110670,58.870900,12.491980,20.207150,77.048040,22.933670,-4.769841,-3.612301,326.581400,13.812000,13.572000,11.127000,0.134680,GALAXY,STARFORMING,1237650795683512507
80.135600,70.843640,367.876300,19.599120,55.355480,510.664700,91.710970,-15.408550,-10.877340,88.712100,12.901000,12.695000,8.763000,0.048086,GALAXY,STARFORMING,1237650796220514489
48.689910,36.945600,147.909000,7.274137,18.083990,351.304500,66.982060,-22.022530,-9.346472,101.350100,13.149000,12.906000,9.260000,0.065183,GALAXY,STARFORMING,1237648720142336237
194.402200,153.680800,563.458600,32.057770,95.847920,1151.211000,257.940900,-38.925330,-18.940330,127.924000,13.091000,12.818000,9.317000,0.064902,GALAXY,STARFORMING,1237648720142336185
140.534800,110.589700,384.858900,30.056240,203.211000,919.356400,194.093000,-59.879910,-25.235250,133.140200,14.109000,13.638000,9.283000,0.114679,GALAXY,nan,1237650795146379447
31.040920,17.307790,56.051570,6.340064,10.572200,144.651800,30.730270,-8.484818,-3.203966,71.449490,13.580000,13.449000,10.118000,0.060614,GALAXY,STARFORMING,1237651799621959813
42.041940,28.548300,64.894130,13.768800,46.406440,131.258900,34.240510,-4.757507,-2.194183,113.018400,13.732000,13.769000,10.907000,0.056548,GALAXY,STARFORMING,1237650795683446947
57.435880,40.396540,149.090700,9.482595,16.896020,435.409100,104.516500,-27.603120,-9.634499,95.077220,13.923000,13.717000,9.705000,0.065110,GALAXY,STARFORMING,1237648720679338241


In [3]:
df[['oiii_h_beta_ratio', 'nii_h_alpha_ratio', 'w2_w3', 'w1_w2', 'el_gas_dens', 'bh_mass']] = np.nan
df[['bpt_class', 'whan_class', 'color_class']] = None

df['oiii_h_beta_ratio'] = df['oiii_5007_flux'] / df['h_beta_flux']
df['nii_h_alpha_ratio'] = df['nii_6584_flux'] / df['h_alpha_flux']
df['w2_w3'] = df['w2mpro'] - df['w3mpro']
df['w1_w2'] = df['w1mpro'] - df['w2mpro']

def el_gas_dens(flux1, flux2, a = 0.4315, b = 2107.0, c = 627.1, Rmin = 0.4375, Rmax = 1.4484):
    'Estimate Electron / Gas Density Using Line Ratios (Sanders et al. 2016)'
    if flux2 == 0.0:
        return np.nan
    R = flux1 / flux2
    if (R >= Rmin) & (R <= Rmax):
        return (c * R - b * a) / (a - R)
    return np.nan

df['el_gas_dens'] = np.vectorize(el_gas_dens)(df['sii_6717_flux'].values, df['sii_6731_flux'].values)

def bh_mass(s, s0 = 200.0, alpha = 8.13, beta = 4.02):
    'Estimate Black Hole Mass Using Galactic Velocity Dispersion (Tremaine et al. 2002)'
    x = np.log10(s / s0)
    y = alpha + beta * x
    return 10**y

df['bh_mass'] = bh_mass(df['oiii_sigma'])

def kewley_discrim_r(r, a = 0.61, b = -0.47, c = 1.19):
    'Classify Galaxies Using Analytical Line Ratio Relation (Kewley et al. 2001)'
    return 10**(a / (np.log10(r) + b) + c)

def kauffmann_discrim_r(r, a = 0.61, b = -0.05, c = 1.3):
    'Classify Galaxies Using Empirical Line Ratio Relation (Kauffmann et al. 2003)'
    return 10**(a / (np.log10(r) + b) + c)

mask = (df['oiii_h_beta_ratio'] <= kewley_discrim_r(df['nii_h_alpha_ratio'])) & (df['oiii_h_beta_ratio'] >= kauffmann_discrim_r(df['nii_h_alpha_ratio']))
df.loc[mask, 'bpt_class'] = 'COMP'

mask = df['oiii_h_beta_ratio'] > kewley_discrim_r(df['nii_h_alpha_ratio'])
df.loc[mask, 'bpt_class'] = 'AGN'

mask = df['oiii_h_beta_ratio'] < kauffmann_discrim_r(df['nii_h_alpha_ratio'])
df.loc[mask, 'bpt_class'] = 'SF'

print(f'\nRatios of PBT Classes:   SF = {100 * len(df[df['bpt_class'] == 'SF']) / len(df):.0f}%  ',
      f'COMP = {100 * len(df[df['bpt_class'] == 'COMP']) / len(df):.0f}%  ',
      f'AGN = {100 * len(df[df['bpt_class'] == 'AGN']) / len(df):.0f}%\n',)

df.loc[(df['nii_h_alpha_ratio'] < 10**(-0.4)) & (-df['h_alpha_eqw'] > 3.0), 'whan_class'] = 'Pure SF'
df.loc[(df['nii_h_alpha_ratio'] > 10**(-0.4)) & (-df['h_alpha_eqw'] > 6.0), 'whan_class'] = 'Strong AGN'
df.loc[(df['nii_h_alpha_ratio'] > 10**(-0.4)) & (-df['h_alpha_eqw'] < 6.0) & (-df['h_alpha_eqw'] > 3.0), 'whan_class'] = 'Weak AGN'
df.loc[-df['h_alpha_eqw'] < 3.0, 'whan_class'] = 'Radio G'
df.loc[(-df['h_alpha_eqw'] < 0.5) & (-df['nii_6584_eqw'] < 0.5), 'whan_class'] = 'Passive G'

print(f'Ratios of WHAN Classes:  Pure SF = {100 * len(df[df['whan_class'] == 'Pure SF']) / len(df):.0f}%  ',
      f'Strong AGN = {100 * len(df[df['whan_class'] == 'Strong AGN']) / len(df):.0f}%  ',
      f'Weak AGN = {100 * len(df[df['whan_class'] == 'Weak AGN']) / len(df):.0f}%  ',
      f'Radio G = {100 * len(df[df['whan_class'] == 'Radio G']) / len(df):.0f}%  ',
      f'Passive G = {100 * len(df[df['bpt_class'] == 'Passive G']) / len(df):.0f}%\n',)

def wedge_up(x, a = 0.315, b = 0.796):
    'Bounds Classification Wedge Upper Border (Mateos et al. 2012)'
    return a * x + b

def wedge_bot(x, a = 0.315, b = -0.222):
    'Bounds Classification Wedge Bottom Border (Mateos et al. 2012)'
    return a * x + b

def wedge_side(x, a = -3.172, b = 7.624):
    'Bounds Classification Wedge Side Border (Mateos et al. 2012)'
    return a * x + b

df.loc[(df['w1_w2'] <= wedge_up(df['w2_w3'])) &
       (df['w1_w2'] >= wedge_bot(df['w2_w3'])) &
       (df['w1_w2'] >= wedge_side(df['w2_w3'])), 'color_class'] = 'AGN'

print(f'Ratio of Color Class:   AGN = {100 * len(df[df['color_class'] == 'AGN']) / len(df):.0f}%\n')

df.head(30).style.hide()
df[(df['color_class'] == 'AGN') & ((df['bpt_class'] == 'AGN') | (df['bpt_class'] == 'COMP')) & ((df['whan_class'] == 'Strong AGN') | (df['whan_class'] == 'Weak AGN'))]#[['bpt_class', 'whan_class', 'color_class']]#.head(100).style.hide()


Ratios of PBT Classes:   SF = 77%   COMP = 16%   AGN = 7%

Ratios of WHAN Classes:  Pure SF = 62%   Strong AGN = 31%   Weak AGN = 4%   Radio G = 3%   Passive G = 0%

Ratio of Color Class:   AGN = 1%



,sii_6717_flux,sii_6731_flux,nii_6584_flux,oi_6300_flux,oiii_5007_flux,h_alpha_flux,h_beta_flux,h_alpha_eqw,nii_6584_eqw,oiii_sigma,...,bestobjid,oiii_h_beta_ratio,nii_h_alpha_ratio,w2_w3,w1_w2,el_gas_dens,bh_mass,bpt_class,whan_class,color_class
112,254.31790,211.64680,708.85730,94.522480,711.99670,960.99240,160.04300,-47.406460,-34.874570,170.61500,...,1237648720143188098,4.448784,0.737630,3.235,1.320,202.097097,7.121453e+07,AGN,Strong AGN,AGN
245,58.77636,49.11408,202.27370,25.799570,344.19030,354.10770,70.86655,-22.121400,-12.251670,116.89790,...,1237654669735756040,4.856880,0.571221,3.092,0.827,207.388599,1.557551e+07,AGN,Strong AGN,AGN
616,491.39700,385.04650,676.47770,231.210700,4045.21100,1598.16400,448.20430,-79.110910,-34.345360,191.86350,...,1237651752388264135,9.025373,0.423284,3.410,1.167,128.879118,1.141534e+08,AGN,Strong AGN,AGN
681,48.50408,34.38325,149.02310,10.796710,56.36860,296.46860,67.79655,-63.614060,-32.842580,126.29850,...,1237654669201375396,0.831438,0.502661,3.014,0.949,25.048573,2.125593e+07,COMP,Strong AGN,AGN
760,43.22939,31.43338,145.70830,9.612429,273.05770,160.29960,36.72366,-11.882330,-10.053430,130.81160,...,1237648722831016156,7.435471,0.908975,3.326,0.850,49.523258,2.447808e+07,AGN,Strong AGN,AGN
764,541.68960,448.94420,1505.97400,112.557300,570.95130,2583.34900,522.04940,-38.688980,-22.416980,117.05110,...,1237648722294210614,1.093673,0.582954,3.256,0.985,196.779147,1.565773e+07,COMP,Strong AGN,AGN
953,311.93190,248.07680,740.05830,125.923300,1184.58800,1288.91400,265.16160,-31.374130,-18.275830,114.35530,...,1237654669203013744,4.467419,0.574172,3.128,1.251,146.088570,1.425771e+07,AGN,Strong AGN,AGN
1220,208.52550,150.05810,509.58880,59.119570,156.54060,883.47520,160.95790,-28.663340,-16.629810,143.65910,...,1237674649921454148,0.972556,0.576800,2.536,0.758,39.381254,3.567292e+07,COMP,Strong AGN,AGN
1358,43.20680,34.96963,140.22330,10.585360,59.02206,234.45120,36.67871,-12.032320,-7.302660,202.94850,...,1237648720687923340,1.609164,0.598092,3.075,0.817,167.098342,1.430706e+08,COMP,Strong AGN,AGN
1402,82.59489,59.36920,212.59500,17.054680,181.91650,486.84980,112.20540,-71.268040,-30.504010,163.70670,...,1237674648848236566,1.621281,0.436675,3.465,0.930,38.286794,6.031239e+07,COMP,Strong AGN,AGN
